In [17]:
# %% Cell 1 — Install libraries
!pip install pandas openpyxl pyarrow requests


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# %% Cell 2 — Imports
import pandas as pd
import numpy as np
import requests

print("Libraries loaded!")

Libraries loaded!


In [19]:
# %% Cell 3 — Load TfL ride data
tfl = pd.read_excel(
    '../data/raw/tfl-daily-cycle-hires.xlsx',
    sheet_name='Data',
    skiprows=5,
    usecols='B:C',
    names=['date', 'rides']
)

# Remove rows where date is empty
tfl = tfl.dropna(subset=['date'])

# Remove rows where rides is not a real number
tfl = tfl[pd.to_numeric(tfl['rides'], errors='coerce').notna()]
tfl['rides'] = tfl['rides'].astype(int)

# Convert to proper datetime
tfl['date'] = pd.to_datetime(tfl['date'])

# Sort oldest to newest
tfl = tfl.sort_values('date').reset_index(drop=True)

print("TfL data loaded!")
print(f"Date range: {tfl['date'].min().date()} to {tfl['date'].max().date()}")
print(f"Number of days: {len(tfl)}")
print(f"Total rides ever: {tfl['rides'].sum():,}")
print()
print(tfl.head())

TfL data loaded!
Date range: 2010-07-30 to 2026-01-31
Number of days: 5665
Total rides ever: 148,350,105

        date  rides
0 2010-07-30   6897
1 2010-07-31   5564
2 2010-08-01   4303
3 2010-08-02   6642
4 2010-08-03   7966


In [ ]:
print("Sample ride values from TfL file:")
print(tfl['rides'].head(20).tolist())
print()
print("Max rides value:", tfl['rides'].max())
print("Min rides value (non-zero):", tfl[tfl['rides'] > 0]['rides'].min())

Sample ride values from TfL file:
[6897, 5564, 4303, 6642, 7966, 7893, 8724, 9797, 6631, 7864, 6191, 4802, 14013, 13080, 12151, 9195, 10928, 15384, 15396, 16062]

Max rides value: 73094
Min rides value (non-zero): 188


In [21]:
# %% Cell 4 — Fetch weather from Open-Meteo API
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 51.5085,
    "longitude": -0.1257,
    "start_date": str(tfl['date'].min().date()),
    "end_date": str(tfl['date'].max().date()),
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "windspeed_10m_max",
        "weathercode"
    ],
    "timezone": "Europe/London"
}

print("Fetching weather from Open-Meteo API...")
response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    print("Success!")
else:
    print("Error:", response.text)

Fetching weather from Open-Meteo API...
Status: 200
Success!


In [22]:
# %% Cell 5 — Parse weather response into DataFrame
data = response.json()

weather = pd.DataFrame({
    'date':          data['daily']['time'],
    'temp_mean':     data['daily']['temperature_2m_mean'],
    'temp_max':      data['daily']['temperature_2m_max'],
    'temp_min':      data['daily']['temperature_2m_min'],
    'precipitation': data['daily']['precipitation_sum'],
    'windspeed':     data['daily']['windspeed_10m_max'],
    'weather_code':  data['daily']['weathercode']
})

weather['date'] = pd.to_datetime(weather['date'])

print("Weather data parsed!")
print(f"Date range: {weather['date'].min().date()} to {weather['date'].max().date()}")
print(f"Number of days: {len(weather)}")
print()
print(weather.head())

Weather data parsed!
Date range: 2010-07-30 to 2026-01-31
Number of days: 5665

        date  temp_mean  temp_max  temp_min  precipitation  windspeed  \
0 2010-07-30       17.6      22.0      12.6            0.0       16.5   
1 2010-07-31       19.1      22.8      16.8            0.1       18.2   
2 2010-08-01       17.6      21.8      13.1            0.0       12.3   
3 2010-08-02       17.4      21.0      14.9            0.2       13.7   
4 2010-08-03       17.5      21.7      12.4            0.0       17.7   

   weather_code  
0             3  
1            51  
2             3  
3            51  
4             3  


In [23]:
# %% Cell 6 — Add readable weather descriptions
def weather_label(code):
    if pd.isna(code):
        return 'Unknown'
    code = int(code)
    if code == 0:
        return 'Clear'
    elif code <= 3:
        return 'Partly cloudy'
    elif code <= 48:
        return 'Fog'
    elif code <= 55:
        return 'Drizzle'
    elif code <= 65:
        return 'Rain'
    elif code <= 75:
        return 'Snow'
    elif code <= 82:
        return 'Showers'
    elif code == 95:
        return 'Thunderstorm'
    else:
        return 'Other'

weather['weather_desc'] = weather['weather_code'].apply(weather_label)

print("Weather descriptions added!")
print(weather['weather_desc'].value_counts())

Weather descriptions added!
weather_desc
Partly cloudy    2297
Drizzle          2283
Rain              878
Snow              139
Clear              68
Name: count, dtype: int64


In [24]:
# %% Cell 7 — Fetch complete UK holidays from gov.uk API
# This gives holidays from 2010 onwards — much better than 
# the CSV which only starts in 2018

print("Fetching UK holidays from gov.uk API...")

url_hol = "https://www.gov.uk/bank-holidays.json"
response_hol = requests.get(url_hol)
print(f"Status: {response_hol.status_code}")

hol_data = response_hol.json()

# We want England and Wales holidays specifically
england_holidays = hol_data['england-and-wales']['events']

holidays = pd.DataFrame(england_holidays)
holidays['date'] = pd.to_datetime(holidays['date'])

print(f"Holidays fetched: {len(holidays)}")
print(f"Date range: {holidays['date'].min().date()} to {holidays['date'].max().date()}")
print()
print(holidays[['title', 'date']].head(10))

# Create a set of holiday dates for fast lookup later
holiday_date_set = set(holidays['date'].dt.date)

Fetching UK holidays from gov.uk API...
Status: 200
Holidays fetched: 83
Date range: 2019-01-01 to 2028-12-26

                    title       date
0          New Year’s Day 2019-01-01
1             Good Friday 2019-04-19
2           Easter Monday 2019-04-22
3  Early May bank holiday 2019-05-06
4     Spring bank holiday 2019-05-27
5     Summer bank holiday 2019-08-26
6           Christmas Day 2019-12-25
7              Boxing Day 2019-12-26
8          New Year’s Day 2020-01-01
9             Good Friday 2020-04-10


In [25]:
# %% Cell 7b — Check holiday coverage by year
print("Holidays per year from gov.uk API:")
holidays_by_year = holidays.groupby(
    holidays['date'].dt.year
).size()
print(holidays_by_year.to_string())
print()
print("Years with fewer than 5 holidays (might be missing data):")
print(holidays_by_year[holidays_by_year < 5])

Holidays per year from gov.uk API:
date
2019     8
2020     8
2021     8
2022    10
2023     9
2024     8
2025     8
2026     8
2027     8
2028     8

Years with fewer than 5 holidays (might be missing data):
Series([], dtype: int64)


In [26]:
# %% Cell 7c — Add missing holidays for 2010-2018 manually
# The gov.uk API only returns from 2019 onwards
# So we manually add all England & Wales bank holidays for 2010-2018

missing_holidays = pd.DataFrame({
    'title': [
        # 2010
        'New Year\'s Day',          'Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day substitute', 'Boxing Day substitute',
        # 2011
        'New Year\'s Day substitute','Good Friday',
        'Easter Monday',            'Royal Wedding',
        'Early May bank holiday',   'Spring bank holiday',
        'Summer bank holiday',      'Christmas Day substitute',
        'Boxing Day substitute',
        # 2012
        'New Year\'s Day substitute','Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Diamond Jubilee',
        'Summer bank holiday',      'Christmas Day',
        'Boxing Day',
        # 2013
        'New Year\'s Day',          'Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day',            'Boxing Day',
        # 2014
        'New Year\'s Day',          'Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day',            'Boxing Day',
        # 2015
        'New Year\'s Day',          'Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day',            'Boxing Day substitute',
        # 2016
        'New Year\'s Day',          'Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day substitute', 'Boxing Day substitute',
        # 2017
        'New Year\'s Day substitute','Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day',            'Boxing Day',
        # 2018
        'New Year\'s Day',          'Good Friday',
        'Easter Monday',            'Early May bank holiday',
        'Spring bank holiday',      'Summer bank holiday',
        'Christmas Day',            'Boxing Day',
    ],
    'date': pd.to_datetime([
        # 2010
        '2010-01-01', '2010-04-02',
        '2010-04-05', '2010-05-03',
        '2010-05-31', '2010-08-30',
        '2010-12-27', '2010-12-28',
        # 2011
        '2011-01-03', '2011-04-22',
        '2011-04-25', '2011-04-29',
        '2011-05-02', '2011-05-30',
        '2011-08-29', '2011-12-26',
        '2011-12-27',
        # 2012
        '2012-01-02', '2012-04-06',
        '2012-04-09', '2012-05-07',
        '2012-06-04', '2012-06-05',
        '2012-08-27', '2012-12-25',
        '2012-12-26',
        # 2013
        '2013-01-01', '2013-03-29',
        '2013-04-01', '2013-05-06',
        '2013-05-27', '2013-08-26',
        '2013-12-25', '2013-12-26',
        # 2014
        '2014-01-01', '2014-04-18',
        '2014-04-21', '2014-05-05',
        '2014-05-26', '2014-08-25',
        '2014-12-25', '2014-12-26',
        # 2015
        '2015-01-01', '2015-04-03',
        '2015-04-06', '2015-05-04',
        '2015-05-25', '2015-08-31',
        '2015-12-25', '2015-12-28',
        # 2016
        '2016-01-01', '2016-03-25',
        '2016-03-28', '2016-05-02',
        '2016-05-30', '2016-08-29',
        '2016-12-27', '2016-12-28',
        # 2017
        '2017-01-02', '2017-04-14',
        '2017-04-17', '2017-05-01',
        '2017-05-29', '2017-08-28',
        '2017-12-25', '2017-12-26',
        # 2018
        '2018-01-01', '2018-03-30',
        '2018-04-02', '2018-05-07',
        '2018-05-28', '2018-08-27',
        '2018-12-25', '2018-12-26',
    ])
})

# Combine with the API holidays we already have
holidays = pd.concat([missing_holidays, holidays], ignore_index=True)

# Remove any duplicates just in case
holidays = holidays.drop_duplicates(subset='date').sort_values('date')

# Rebuild the holiday date set
holiday_date_set = set(holidays['date'].dt.date)

print(f"Total holidays after adding 2010-2018: {len(holidays)}")
print()
print("Holidays per year — full picture:")
print(holidays.groupby(holidays['date'].dt.year).size().to_string())

Total holidays after adding 2010-2018: 157

Holidays per year — full picture:
date
2010     8
2011     9
2012     9
2013     8
2014     8
2015     8
2016     8
2017     8
2018     8
2019     8
2020     8
2021     8
2022    10
2023     9
2024     8
2025     8
2026     8
2027     8
2028     8


In [27]:
# %% Cell 8 — Merge TfL rides + weather
merged = pd.merge(tfl, weather, on='date', how='inner')

print(f"Merged rides + weather: {len(merged)} rows")
print(f"Date range: {merged['date'].min().date()} to {merged['date'].max().date()}")

Merged rides + weather: 5665 rows
Date range: 2010-07-30 to 2026-01-31


In [28]:
# %% Cell 9 — Add holiday flag and time features
# Holiday flag — check each date against our gov.uk holiday set
merged['is_holiday'] = merged['date'].dt.date.apply(
    lambda d: 1 if d in holiday_date_set else 0
)

# Time features
merged['year']        = merged['date'].dt.year
merged['month']       = merged['date'].dt.month
merged['day_of_week'] = merged['date'].dt.dayofweek  # 0=Mon, 6=Sun
merged['is_weekend']  = (merged['day_of_week'] >= 5).astype(int)

# Season from month
merged['season'] = merged['month'].map({
    12:4, 1:4, 2:4,
    3:1,  4:1, 5:1,
    6:2,  7:2, 8:2,
    9:3, 10:3, 11:3
})
merged['season_name'] = merged['season'].map({
    1:'Spring', 2:'Summer', 3:'Autumn', 4:'Winter'
})

print("All columns added!")
print()
print(merged[['date','rides','temp_mean','precipitation',
              'is_holiday','is_weekend','season_name']].head(10))

All columns added!

        date  rides  temp_mean  precipitation  is_holiday  is_weekend  \
0 2010-07-30   6897       17.6            0.0           0           0   
1 2010-07-31   5564       19.1            0.1           0           1   
2 2010-08-01   4303       17.6            0.0           0           1   
3 2010-08-02   6642       17.4            0.2           0           0   
4 2010-08-03   7966       17.5            0.0           0           0   
5 2010-08-04   7893       16.1           12.8           0           0   
6 2010-08-05   8724       15.7            0.0           0           0   
7 2010-08-06   9797       15.6            1.0           0           0   
8 2010-08-07   6631       17.7            3.3           0           1   
9 2010-08-08   7864       17.4            0.0           0           1   

  season_name  
0      Summer  
1      Summer  
2      Summer  
3      Summer  
4      Summer  
5      Summer  
6      Summer  
7      Summer  
8      Summer  
9      Summer  


In [29]:
# %% Cell 9b — Flag and handle known anomalies
# 10-11 September 2022: zero rides due to national mourning
# after Queen Elizabeth II died on 8 September 2022.
# These are real events, not data errors. We keep the rows but
# add a flag so we can choose to exclude them from analysis if needed.

anomaly_dates = ['2022-09-10', '2022-09-11']

merged['is_anomaly'] = merged['date'].dt.strftime('%Y-%m-%d').isin(
    anomaly_dates
).astype(int)

merged['anomaly_reason'] = merged['date'].dt.strftime('%Y-%m-%d').map({
    '2022-09-10': 'National mourning - Queen Elizabeth II',
    '2022-09-11': 'National mourning - Queen Elizabeth II'
}).fillna('')

print("Anomaly days flagged:")
print(merged[merged['is_anomaly'] == 1][
    ['date', 'rides', 'anomaly_reason']
])

Anomaly days flagged:
           date  rides                          anomaly_reason
4425 2022-09-10      0  National mourning - Queen Elizabeth II
4426 2022-09-11      0  National mourning - Queen Elizabeth II


In [30]:
# %% Cell 10 — Sanity check
print("=== FINAL DATASET SUMMARY ===")
print(f"Total rows (days):         {len(merged)}")
print(f"Date range:                {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"Missing values anywhere:   {merged.isnull().sum().sum()}")
print(f"Holiday days in dataset:   {merged['is_holiday'].sum()}")
print(f"Weekend days:              {merged['is_weekend'].sum()}")
print(f"Average rides per day:     {merged['rides'].mean():,.0f}")
print(f"Busiest day ever:          {merged['rides'].max():,} rides on {merged.loc[merged['rides'].idxmax(), 'date'].date()}")
print(f"Quietest day ever:         {merged['rides'].min():,} rides on {merged.loc[merged['rides'].idxmin(), 'date'].date()}")

=== FINAL DATASET SUMMARY ===
Total rows (days):         5665
Date range:                2010-07-30 to 2026-01-31
Missing values anywhere:   0
Holiday days in dataset:   129
Weekend days:              1619
Average rides per day:     26,187
Busiest day ever:          73,094 rides on 2015-07-09
Quietest day ever:         0 rides on 2022-09-10


In [31]:
# %% Cell 11 — Save everything
import os
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Save raw weather so you never need to call the API again
weather.to_csv('../data/raw/weather_london.csv', index=False)
print("Saved: data/raw/weather_london.csv")

# Save complete holidays
holidays.to_csv('../data/raw/uk_holidays_complete.csv', index=False)
print("Saved: data/raw/uk_holidays_complete.csv")

# Save the merged dataset — this is what all future notebooks load
merged.to_parquet('../data/processed/merged.parquet', index=False)
print("Saved: data/processed/merged.parquet")

print()
print(f"Done! {len(merged)} days of data ready for analysis.")

Saved: data/raw/weather_london.csv
Saved: data/raw/uk_holidays_complete.csv
Saved: data/processed/merged.parquet

Done! 5665 days of data ready for analysis.


In [32]:
# Remove the anomaly days from the dataset entirely
# Keeping 0-ride days would distort averages and confuse the model
# We document why we removed them in the anomaly flag above

rows_before = len(merged)
merged = merged[merged['is_anomaly'] == 0].reset_index(drop=True)
rows_after = len(merged)

print(f"Removed {rows_before - rows_after} anomaly rows")
print(f"Dataset now has {rows_after} rows")

Removed 2 anomaly rows
Dataset now has 5663 rows
